# Lecture 15 (Decorators)

## Exercise 15.1 (integer rounding)

Create a function decorator `@integer_round` that can be applied to a function
returning a float. The decorated function should apply the builtin function
`round` to the returned float value.

An example using the decorator is

    @integer_round
    def f(x):
        return x ** 0.5  # square root

where `f(30)` should return `5`, and `f(31)` should return `6`.

In [1]:
def integer_round(f):
    def wrapper(*args, **kwargs):
        return round(f(*args, **kwargs))
    return wrapper

@integer_round
def f(x):
    return x ** 0.5  # square root


print(f'{f(30) = }')
print(f'{f(31) = }')

f(30) = 5
f(31) = 6


## Exercise 15.2 (decorator returns int)

Create a function decorator `@enforce_integer_return` that checks if a function returns a value of type `int` and raises an `AssertionError` if the result is of a different type. The decorator could e.g. be used as in the example below.

    > @enforce_integer_return
      def my_sum(x, y):
          return x + y

    > my_sum(19, 23)
    42

    > my_sum(1, 2.5)
    ...
    AssertionError: result must be int

In [ ]:
def enforce_integer_return(f):
    def wrapper(*args, **kwargs):
        result = f(*args, **kwargs)
        assert isinstance(result, int), f'result is {result}, must be int'
        return result

    return wrapper


@enforce_integer_return
def my_sum(x, y):
    return x + y

# my_sum = enforce_integer_return(my_sum)

print(my_sum(19, 23))
print(my_sum(2, 3.5))

## Exercise 15.3 (box it)

Create a decorator `@BoxIt` for functions returning a string. The output of a function returning a string should be
converted into a new string, with the output put into a 'box' (see example below).  You can assume that the string returned by a function only consist of a single line of text, i.e. the string does not contain `'\n'`.

    @BoxIt
    def plus(x, y):
        return f'The sum of {x} and {y} is {x + y}'

    print(plus(3,4))

should print the following

    ---------------------------
    | The sum of 3 and 4 is 7 |
    ---------------------------

In [ ]:
def BoxIt(f):
    def wrapper(*args, **kwargs):
        result = f(*args, **kwargs)
        txt = f'| {result} |'
        width = len(txt)
        line = '-' * width
        return line + '\n' + txt + '\n' + line

    return wrapper


@BoxIt
def plus(x, y):
    return f'The sum of {x} and {y} is {x + y}'


print(plus(3,4))

## Exercise 15.4 (multiline box it) [optional]

Generalize your solution to Exercise 15.3, so that it can handle multiline output.

    @BoxIt
    def test():
        return 'This\n   is a\n      test'

    @BoxIt
    @BoxIt
    def plus(x, y):
        return f'{x} + {y} = {x + y}'

    print(test())
    print(plus(3, 4))

should print the following

    +------------+
    | This       |
    |    is a    |
    |       test |
    +------------+
    +---------------+
    | +-----------+ |
    | | 3 + 4 = 7 | |
    | +-----------+ |
    +---------------+

In [15]:
def BoxIt(f):
    def wrapper(*args, **kwargs):
        result = f(*args, **kwargs)
        txt = str(result)
        lines = txt.split('\n')
        width = max([len(line) for line in lines])
        #lines = ['| ' + line + ' ' * (width - len(line)) + ' |\n' for line in lines]
        #lines = ['| ' + line.ljust(width) + ' |\n' for line in lines]
        lines = [f'| {line:{width}} |\n' for line in lines]
        headerline = '+' + '-' * (width + 2) + '+'
        return  headerline + '\n' + ''.join(lines) + headerline

    return wrapper

@BoxIt
def test():
    return 'This\n   is a\n      test'


@BoxIt
@BoxIt
def plus(x, y):
    return f'{x} + {y} = {x + y}'


print(test())
print(plus(3,4))

+------------+
| This       |
|    is a    |
|       test |
+------------+
+---------------+
| +-----------+ |
| | 3 + 4 = 7 | |
| +-----------+ |
+---------------+


## Exercise 15.5** (decorator composition) [optional]

If you repeatedly apply the same set of decorators to your functions, it might be convenient to compose these decorators into one decorator.

Recall that:

    @dec3
    @dec2
    @dec1
    def my_func(*args):
        ...

is equivalent to

    def my_func(*args):
        ...

    my_func = dec3(dec2(dec1(my_func)))

One could compose the three decorators into one decorator `dec_composed` as

    def dec_composed(f):
        return dec3(dec2(dec1(f)))

and using this new decorator as

    @dec_composed
    def my_func(*args):
        ...

In this exercise you should make a function `compose` that takes a sequence of decorator functions, and composes them into one new decorator. The above decorator composition could then have been written as:

    dec_composed = compose(dec3, dec2, dec1)

Alternatively one could use `compose` as a decorator with arguments

    @compose(dec3, dec2, dec1)
    def my_func(*args):
        ...

The below example uses the `compose` function in the two different ways.

    def double(f):
        def wrapper(*args):
            return 2 * f(*args)

        return wrapper

    def add_three(f):
        def wrapper(*args):
            return 3 + f(*args)

        return wrapper

    @compose(double, double, add_three)
    def seven():
        return 7

    print(seven())  # prints 40

    my_decorator = compose(double, double, add_three)

    @my_decorator
    def seven():
        return 7

    print(seven())  # prints 40

_Note_. The function `compose` mathematically corresponds to [function composition](https://en.wikipedia.org/wiki/Function_composition).

In [1]:
def compose(*decorators):
    def decorator(f):
        for dec in reversed(decorators):
            f = dec(f)
        return f

    return decorator


def double(f):
    def wrapper(*args):
        return 2 * f(*args)

    return wrapper


def add_three(f):
    def wrapper(*args):
        return 3 + f(*args)

    return wrapper


@compose(double, double, add_three)
def seven():
    return 7


print(seven())  # prints 40

my_decorator = compose(double, double, add_three)


@my_decorator
def seven():
    return 7


print(seven())  # prints 40

40
40


## Exercise 15.6** (numba) [optional]

The module `numba` ([numba.pydata.org](https://numba.pydata.org/)) provides a decorator `jit` that allows ''[Just In Time compilation](https://en.wikipedia.org/wiki/Just-in-time_compilation)'' of a Python function, provided the function uses sufficiently simple Python code. The first time a `jit` decorated function is called, the function is translated into efficient low level code, that will be used for each of the subsequent calls.

Try to install `numba` and apply the decorator `@numba.jit(nopython=True)` to a function, e.g., your Newton-Raphson square root computation from Exercise 2.7, or a function computing the sum of all primes in the range 2 to _n_.

Measure the computation time with and without using Numba, e.g., using `time.time` and by repeating the computation sufficiently many times to make the computation time measurable, say 1 second. Note that the first call to a `jit` decorated function can take significantly more time than subsequent calls.

_Note_: Using `numba` optimally is nontrivial, and requires some knowledge about the inner workings of Numba and computer architecture. For functions doing numeric calculations using loops, speedups of the order of a factor 25 are not uncommon by applying just in time compilation.

In [19]:
import numba
import math
from time import time

# @numba.jit(nopython=True)
def sqrt(n):  # Newton-Raphson (n >= 1)
    last = n + 1.0
    r = n
    while r < last:
        last = r
        r = (r + n / r) / 2
    return r

numba_sqrt = numba.jit(nopython=True)(sqrt)

for name, f in [
    ('math.sqrt', math.sqrt),
    ('sqrt', sqrt),
    ('numba_sqrt', numba_sqrt),
    ('numba_sqrt', numba_sqrt),
    ('lambda x: x ** 0.5', lambda x: x ** 0.5)
]:
    x = math.pi
    start = time()
    for _ in range(1_000_000):
        root = f(x)
    end = time()
    print(f'{name}({x}) = {root} [{end - start:.3f} sec]')

math.sqrt(3.141592653589793) = 1.7724538509055159 [0.203 sec]
sqrt(3.141592653589793) = 1.7724538509055159 [0.585 sec]
numba_sqrt(3.141592653589793) = 1.7724538509055159 [0.254 sec]
numba_sqrt(3.141592653589793) = 1.7724538509055159 [0.192 sec]
lambda x: x ** 0.5(3.141592653589793) = 1.7724538509055159 [0.179 sec]


In [ ]:
import numba
from time import time

# @numba.jit(nopython=True)
def prime_sum(n):
    prime = [True] * (n + 1)
    for f in range(2, n + 1):
        for x in range(2 * f, n + 1, f):
            prime[x] = False
    return sum([i for i in range(2, n + 1) if prime[i]])

numba_prime_sum = numba.jit(nopython=True)(prime_sum)
numba_prime_sum.__name__ += ' (numba)'

def timeit(f, n=2000000):
    start = time()
    answer = f(n)
    end = time()
    print(f.__name__, answer, end - start)

timeit(prime_sum)
timeit(prime_sum)
timeit(prime_sum)
timeit(numba_prime_sum)  # slow, first call does JIT compilation
timeit(numba_prime_sum)
timeit(numba_prime_sum)
timeit(numba_prime_sum)

prime_sum 142913828922 1.6430082321166992
prime_sum 142913828922 1.5988218784332275
prime_sum 142913828922 1.5987930297851562
prime_sum (numba) 142913828922 0.3237013816833496
prime_sum (numba) 142913828922 0.06289005279541016
prime_sum (numba) 142913828922 0.05318713188171387
prime_sum (numba) 142913828922 0.0602114200592041
